# Feature Engineering

In [ ]:
#importando bibliotecas
import pandas as pd
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
import plotly.figure_factory as ff 
from plotly.subplots import make_subplots

import pandas as pd
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

pd.set_option('display.max_columns', None)

: 

In [ ]:
#importando dados
df = pd.read_csv('../data/curated/data.csv')

In [ ]:
df_airports_profile = df.groupby('ORIGIN_AIRPORT').agg({
    'IS_DELAYED': ['mean', 'count']
}).reset_index()
df_airports_profile.columns = ['ORIGIN_AIRPORT', 'DELAY_RATE', 'FLIGHT_VOLUME']

kmeans = KMeans(n_clusters=4, random_state=42)
df_airports_profile['AIRPORT_PROFILE'] = kmeans.fit_predict(df_airports_profile[['DELAY_RATE', 'FLIGHT_VOLUME']])

In [ ]:
fig = px.scatter(
    df_airports_profile, 
    x='FLIGHT_VOLUME', 
    y='DELAY_RATE',
    color='AIRPORT_PROFILE',
    hover_name='ORIGIN_AIRPORT',
    title='Agrupamento de Aeroportos: Volume vs. Taxa de Atraso',
    labels={'FLIGHT_VOLUME': 'Volume Total de Voos', 'DELAY_RATE': 'Taxa de Atraso (0 a 1)'},
    template='plotly_white',
    color_discrete_sequence=px.colors.qualitative.Safe
)

fig.show()

A análise de agrupamento revelou que o volume de operações é o principal fator de diferenciação entre os aeroportos. Enquanto grandes centros (hubs) mantêm uma taxa de atraso constante entre 15% e 25%, aeroportos de pequeno porte apresentam alta volatilidade, sendo mais suscetíveis a picos de atraso superiores a 40%, provavelmente devido à menor resiliência operacional.

In [ ]:
df.AIRLINE_NAME.unique()

<StringArray>
[        'Alaska Airlines Inc.',       'American Airlines Inc.',
              'US Airways Inc.',         'Delta Air Lines Inc.',
             'Spirit Air Lines',        'United Air Lines Inc.',
       'Hawaiian Airlines Inc.',              'JetBlue Airways',
        'Skywest Airlines Inc.',  'Atlantic Southeast Airlines',
       'Frontier Airlines Inc.',       'Southwest Airlines Co.',
 'American Eagle Airlines Inc.',               'Virgin America']
Length: 14, dtype: str

In [ ]:
df['AIRLINE'].unique()

<StringArray>
['AS', 'AA', 'US', 'DL', 'NK', 'UA', 'HA', 'B6', 'OO', 'EV', 'F9', 'WN', 'MQ',
 'VX']
Length: 14, dtype: str

In [ ]:
df_airline_profile = df.groupby('AIRLINE_NAME').agg({
    'IS_DELAYED': ['mean', 'count']
}).reset_index()
df_airline_profile.columns = ['AIRLINE_NAME', 'DELAY_RATE', 'FLIGHT_VOLUME']

kmeans = KMeans(n_clusters=4, random_state=42)
df_airline_profile['AIRLINE_PROFILE'] = kmeans.fit_predict(df_airline_profile[['DELAY_RATE', 'FLIGHT_VOLUME']])

fig = px.scatter(
    df_airline_profile, 
    x='FLIGHT_VOLUME', 
    y='DELAY_RATE',
    color='AIRLINE_PROFILE',
    hover_name='AIRLINE_NAME',
    title='Agrupamento de Aeroportos: Volume vs. Taxa de Atraso',
    labels={'FLIGHT_VOLUME': 'Volume Total de Voos', 'DELAY_RATE': 'Taxa de Atraso (0 a 1)'},
    template='plotly_white',
    color_discrete_sequence=px.colors.qualitative.Safe
)

fig.show()

In [ ]:
def get_time_of_day(raw):
    hour = raw // 100
    if 0 <= hour < 6:
        return 1 #madrugada
    elif 6 <= hour < 12:
        return 2 #manhã
    elif 12 <= hour < 18:
        return 3 #tarde
    else:
        return 4 #noite

# Aplicando com um novo nome de coluna mais apropriado
df['TIME_OF_DAY'] = df['SCHEDULED_DEPARTURE'].apply(get_time_of_day)

In [ ]:
seasons = {
    12: 1, 1: 1, 2: 1,
    3: 2, 4: 2, 5: 2,
    6: 3, 7: 3, 8: 3,
    9: 4, 10: 4, 11: 4
}

df['SEASON'] = df['MONTH'].map(seasons)

In [ ]:
df

,YEAR,MONTH,DAY,DAY_OF_WEEK,AIRLINE,FLIGHT_NUMBER,TAIL_NUMBER,ORIGIN_AIRPORT,DESTINATION_AIRPORT,SCHEDULED_DEPARTURE,DEPARTURE_TIME,DEPARTURE_DELAY,TAXI_OUT,WHEELS_OFF,SCHEDULED_TIME,ELAPSED_TIME,AIR_TIME,DISTANCE,WHEELS_ON,TAXI_IN,SCHEDULED_ARRIVAL,ARRIVAL_TIME,ARRIVAL_DELAY,DIVERTED,CANCELLED,AIR_SYSTEM_DELAY,SECURITY_DELAY,AIRLINE_DELAY,LATE_AIRCRAFT_DELAY,WEATHER_DELAY,AIRLINE_NAME,ORIGIN_AIRPORT_NAME,ORIGIN_CITY,ORIGIN_STATE,ORIGIN_COUNTRY,ORIGIN_LATITUDE,ORIGIN_LONGITUDE,DEST_AIRPORT_NAME,DEST_CITY,DEST_STATE,DEST_COUNTRY,DEST_LATITUDE,DEST_LONGITUDE,IS_DELAYED,PERIOD,SEASON
0,2015,1,1,4,AS,98,N407AS,ANC,SEA,5,2354.0,-11.0,21.0,15.0,205.0,194.0,169.0,1448,404.0,4.0,430,408.0,-22.0,0,0,0.0,0.0,0.0,0.0,0.0,Alaska Airlines Inc.,Ted Stevens Anchorage International Airport,Anchorage,AK,USA,61.17432,-149.99619,Seattle-Tacoma International Airport,Seattle,WA,USA,47.44898,-122.30931,0,1,1
1,2015,1,1,4,AA,2336,N3KUAA,LAX,PBI,10,2.0,-8.0,12.0,14.0,280.0,279.0,263.0,2330,737.0,4.0,750,741.0,-9.0,0,0,0.0,0.0,0.0,0.0,0.0,American Airlines Inc.,Los Angeles International Airport,Los Angeles,CA,USA,33.94254,-118.40807,Palm Beach International Airport,West Palm Beach,FL,USA,26.68316,-80.09559,0,1,1
2,2015,1,1,4,US,840,N171US,SFO,CLT,20,18.0,-2.0,16.0,34.0,286.0,293.0,266.0,2296,800.0,11.0,806,811.0,5.0,0,0,0.0,0.0,0.0,0.0,0.0,US Airways Inc.,San Francisco International Airport,San Francisco,CA,USA,37.61900,-122.37484,Charlotte Douglas International Airport,Charlotte,NC,USA,35.21401,-80.94313,0,1,1
3,2015,1,1,4,AA,258,N3HYAA,LAX,MIA,20,15.0,-5.0,15.0,30.0,285.0,281.0,258.0,2342,748.0,8.0,805,756.0,-9.0,0,0,0.0,0.0,0.0,0.0,0.0,American Airlines Inc.,Los Angeles International Airport,Los Angeles,CA,USA,33.94254,-118.40807,Miami International Airport,Miami,FL,USA,25.79325,-80.29056,0,1,1
4,2015,1,1,4,AS,135,N527AS,SEA,ANC,25,24.0,-1.0,11.0,35.0,235.0,215.0,199.0,1448,254.0,5.0,320,259.0,-21.0,0,0,0.0,0.0,0.0,0.0,0.0,Alaska Airlines Inc.,Seattle-Tacoma International Airport,Seattle,WA,USA,47.44898,-122.30931,Ted Stevens Anchorage International Airport,Anchorage,AK,USA,61.17432,-149.99619,0,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5221995,2015,12,31,4,B6,688,N657JB,LAX,BOS,2359,2355.0,-4.0,22.0,17.0,320.0,298.0,272.0,2611,749.0,4.0,819,753.0,-26.0,0,0,0.0,0.0,0.0,0.0,0.0,JetBlue Airways,Los Angeles International Airport,Los Angeles,CA,USA,33.94254,-118.40807,Gen. Edward Lawrence Logan International Airport,Boston,MA,USA,42.36435,-71.00518,0,4,1
5221996,2015,12,31,4,B6,745,N828JB,JFK,PSE,2359,2355.0,-4.0,17.0,12.0,227.0,215.0,195.0,1617,427.0,3.0,446,430.0,-16.0,0,0,0.0,0.0,0.0,0.0,0.0,JetBlue Airways,John F. Kennedy International Airport (New Yor...,New York,NY,USA,40.63975,-73.77893,Mercedita Airport,Ponce,PR,USA,18.00830,-66.56301,0,4,1
5221997,2015,12,31,4,B6,1503,N913JB,JFK,SJU,2359,2350.0,-9.0,17.0,7.0,221.0,222.0,197.0,1598,424.0,8.0,440,432.0,-8.0,0,0,0.0,0.0,0.0,0.0,0.0,JetBlue Airways,John F. Kennedy International Airport (New Yor...,New York,NY,USA,40.63975,-73.77893,Luis Muñoz Marín International Airport,San Juan,PR,USA,18.43942,-66.00183,0,4,1
5221998,2015,12,31,4,B6,333,N527JB,MCO,SJU,2359,2353.0,-6.0,10.0,3.0,161.0,157.0,144.0,1189,327.0,3.0,340,330.0,-10.0,0,0,0.0,0.0,0.0,0.0,0.0,JetBlue Airways,Orlando International Airport,Orlando,FL,USA,28.42889,-81.31603,Luis Muñoz Marín International Airport,San Juan,PR,USA,18.43942,-66.00183,0,4,1


Correlação entre variáveis numéricas

In [ ]:
df['SCHEDULED_DEPARTURE_HOUR'] = df['SCHEDULED_DEPARTURE'] // 100

df = df.merge(df_airports_profile[['ORIGIN_AIRPORT', 'AIRPORT_PROFILE']], on='ORIGIN_AIRPORT', how='left')
df = df.merge(df_airline_profile[['AIRLINE_NAME', 'AIRLINE_PROFILE']], on='AIRLINE_NAME', how='left')

num_cols = [
    'MONTH', 'DAY_OF_WEEK', 'SCHEDULED_DEPARTURE_HOUR', 'TIME_OF_DAY', 'SEASON', 'DISTANCE', 'AIRPORT_PROFILE', 'AIRLINE_PROFILE', 'IS_DELAYED'
]

corr = df[num_cols].corr()

In [ ]:
fig = px.imshow(
    corr,
    text_auto='.2f',
    aspect='auto',
    color_continuous_scale='RdBu_r',
    zmin=-1, 
    zmax=1,
    title='Matriz de Correlação',
    labels=dict(color='Correlação')
)
fig.update_layout(
    height=600,
    template='plotly_white'
)
fig.show()

In [ ]:
features = [
    'MONTH', 'DAY_OF_WEEK', 'SCHEDULED_DEPARTURE_HOUR', 'TIME_OF_DAY', 'SEASON', 'DISTANCE', 'AIRPORT_PROFILE', 'AIRLINE_PROFILE', 
]

target = 'IS_DELAYED'

df = df[features + [target]].copy()

df_atrasados = df[df[target] == 1]
df_pontuais = df[df[target] == 0]

n_atrasados = len(df_atrasados)
df_pontuais_bal = df_pontuais.sample(n=n_atrasados, random_state=42)

df_balanced = pd.concat([df_atrasados, df_pontuais_bal])
df_balanced = df_balanced.sample(frac=1, random_state=42).reset_index(drop=True)

print(df_balanced.shape[0])

1929746


In [ ]:
df_balanced.SCHEDULED_DEPARTURE_HOUR.unique()

array([19, 17, 13, 12, 10, 20,  7,  6, 14, 15, 16,  9,  8, 11, 18,  5, 21,
       22, 23,  0,  3,  1,  4,  2])

In [ ]:
df_balanced.head()

,MONTH,DAY_OF_WEEK,SCHEDULED_DEPARTURE_HOUR,PERIOD,SEASON,DISTANCE,AIRPORT_PROFILE,AIRLINE_PROFILE,IS_DELAYED
0,7,1,19,4,3,1790,3,3,1
1,11,2,17,3,4,89,0,1,1
2,1,5,19,4,1,212,2,0,1
3,2,7,13,3,1,831,2,1,0
4,11,7,12,3,4,859,0,2,0


In [ ]:
scaler = StandardScaler()

cols_to_normalize = df_balanced.columns[:-1] 

df_normalized = df_balanced.copy()
df_normalized[cols_to_normalize] = scaler.fit_transform(df_normalized[cols_to_normalize])

In [ ]:
df_normalized.head()

,MONTH,DAY_OF_WEEK,SCHEDULED_DEPARTURE_HOUR,PERIOD,SEASON,DISTANCE,AIRPORT_PROFILE,AIRLINE_PROFILE,IS_DELAYED
0,0.253167,-1.456333,1.135593,1.397838,0.625861,1.580078,1.428664,1.432931,1
1,1.436686,-0.953642,0.716535,0.161628,1.588467,-1.216751,-1.374316,-0.434042,1
2,-1.522113,0.554430,1.135593,1.397838,-1.299350,-1.014511,0.494337,-1.367529,1
3,-1.226233,1.559811,-0.121581,0.161628,-1.299350,0.003265,0.494337,-0.434042,0
4,1.436686,1.559811,-0.331110,0.161628,1.588467,0.049304,-1.374316,0.499444,0


Exportando features

In [ ]:
df_normalized.to_csv('../data/curated/features.csv', index=False)